# Crawl và đo benchmark arXiv trên Colab CPU

Mỗi thành viên chạy **toàn bộ pipeline cho owner của mình** trên Colab CPU, dùng cùng commit đã kiểm chứng. Dữ liệu thu được dùng cho đồ án; benchmark đo ngay trong cùng lượt chạy. Chọn **Runtime → Change runtime type → Hardware accelerator: None (CPU)**.

Đã kiểm tra pilot và resume trên Colab. Cấu hình mặc định bên dưới là **toàn dải, chạy theo đợt 500 task mới**, không phải 500 bài. Sau mỗi đợt, chạy các cell kiểm tra và sao lưu rồi tiếp tục bằng `resume` với measurement ID mới. Các phép đo theo đợt phải được báo cáo riêng, không mô tả một đợt là toàn bộ lượt crawl hoàn chỉnh.

Corpus/SQLite ở ổ VM. Drive chung `/content/drive/MyDrive/arXiv-data` chỉ nhận snapshot đã dừng và số liệu. Mỗi tài khoản phải có quyền ghi và shortcut đúng thư mục chung; notebook kiểm tra quyền sau mount. [Colab FAQ](https://research.google.com/colaboratory/faq.html) mô tả vòng đời VM và giới hạn phiên; hiện chưa có backup nền trong lúc runner đang chạy.


## 1. Điền cấu hình

Code pipeline được ghim vào commit `af99d119a3dcc0f8f25bfa2c9d5a6b6d9d7bb218` đã kiểm chứng. Owner: `trong`, `khang`, `hoang`, `man-nhi`, `huynh-nhi`. Mỗi người đổi `OWNER`; `RUN_ID` tự tạo riêng. `full` bỏ cả hai giới hạn mẫu, dù các biến `MAX_CANDIDATES`/`MAX_PAPERS` vẫn có trong cell để dùng khi chọn `pilot`.

`STOP_AFTER_TASKS=500` dừng sau 500 task mới trong invocation, không phải số bài; discovery cả dải là một task và có thể lâu. Khi tiếp tục, giữ `RUN_ID`, đổi `RUN_MODE` thành `resume`, đổi `MEASUREMENT_ID` thành `resume-01`, `resume-02`… Không dùng lại measurement ID đã có số đo.

`SNAPSHOT_MAX_FILES=500_000` là giới hạn file cho mọi thao tác snapshot trong notebook. Dùng cùng giá trị khi leader verify/unpack/restore bằng CLI; `run.json` bàn giao ghi giá trị đã dùng. Các giới hạn 2 GiB/file và 20 GiB tổng byte giải nén của CLI vẫn giữ nguyên; chưa kiểm chứng thực tế một archive 500.000 file.

Drive chung của nhóm đã được tạo. Thêm [shortcut vào My Drive](https://support.google.com/drive/answer/9700156) nếu thư mục nằm trong Shared with me. Kiểm tra đường dẫn `TEAM_DRIVE_DIR` trên từng tài khoản; không tạo các thư mục riêng cùng tên rồi coi là đã chia sẻ.


In [ ]:
from pathlib import Path

COMMIT_SHA = "af99d119a3dcc0f8f25bfa2c9d5a6b6d9d7bb218"  # SHA 40 ký tự do leader cung cấp; không dùng tên nhánh/main
OWNER = "trong"
RUN_ID = f"m1-colab-{OWNER}-01"
RUN_MODE = "full"  # pilot | full | resume
MEASUREMENT_ID = "cold-01"  # resume dùng resume-01, resume-02, ...
MAX_CANDIDATES = 40
MAX_PAPERS = 10
STOP_AFTER_TASKS = 500  # task mới mỗi đợt; không phải số bài
WORK_ROOT = Path("/content/arxiv-work")
CORPUS_PATH = ""  # điền đường dẫn corpus đã restore khi tiếp tục ở vị trí khác
RESTORE_BUNDLE = ""  # đường dẫn zip trên Drive khi cần khôi phục trong phiên mới
TEAM_DRIVE_DIR = "/content/drive/MyDrive/arXiv-data"  # đường dẫn thư mục Drive chung đã có; không phải URL chia sẻ
SNAPSHOT_MAX_FILES = 500_000  # dùng chung cho export/verify/restore
REPO_DIR = Path(f"/content/arxiv-code-{COMMIT_SHA[:12]}")


## 2. Lấy đúng code và cài dependencies

Cell này clone repo công khai nếu chưa có; không xóa checkout hoặc dữ liệu cũ. Nếu checkout không khớp commit hoặc code pipeline bị sửa, cell dừng để tránh gán sai provenance.

Nếu cần API key Semantic Scholar, đặt `S2_API_KEY` trong môi trường bằng cách quản lý secrets của bạn; không ghi key vào notebook hoặc in ra output.


In [ ]:
import importlib.metadata
import json
import os
import platform
import re
import subprocess
import sys

if not re.fullmatch(r"[0-9a-f]{40}", COMMIT_SHA):
    raise ValueError("Điền COMMIT_SHA 40 ký tự do leader cung cấp trước khi chạy.")
if OWNER not in {"trong", "khang", "hoang", "man-nhi", "huynh-nhi"}:
    raise ValueError("OWNER không thuộc nhóm.")
if RUN_MODE not in {"pilot", "full", "resume"}:
    raise ValueError("RUN_MODE phải là pilot, full hoặc resume.")
for name, value in {"RUN_ID": RUN_ID, "MEASUREMENT_ID": MEASUREMENT_ID}.items():
    if not isinstance(value, str) or not re.fullmatch(r"[A-Za-z0-9_-]+", value):
        raise ValueError(f"{name} chỉ gồm chữ, số, dấu gạch ngang và gạch dưới.")
if sys.version_info < (3, 10):
    raise RuntimeError("Pipeline cần Python >= 3.10.")
if WORK_ROOT.resolve().is_relative_to(Path("/content/drive").resolve()):
    raise ValueError("WORK_ROOT phải ở ổ làm việc local, không đặt SQLite trong Drive.")

REPO_URL = "https://github.com/ChickenDev-cyber/arxiv-data-science.git"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", COMMIT_SHA], check=True)

def git_output(*args):
    return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

def ensure_release():
    if Path(git_output("rev-parse", "--show-toplevel")).resolve() != REPO_DIR.resolve():
        raise RuntimeError("REPO_DIR không phải gốc checkout đã chọn.")
    origin = git_output("remote", "get-url", "origin").removesuffix(".git").lower()
    if origin != REPO_URL.removesuffix(".git").lower():
        raise RuntimeError("Checkout không thuộc repo mã nguồn của nhóm.")
    head = git_output("rev-parse", "HEAD")
    if head != COMMIT_SHA or git_output("status", "--porcelain", "--", "src/arxiv_pipeline", "src/requirements.txt"):
        raise RuntimeError("Commit không khớp hoặc source pipeline đang có sửa đổi chưa commit.")
    return head

ensure_release()
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "src/requirements.txt")], check=True)
PIPELINE_ENV = {**os.environ, "PYTHONPATH": str(REPO_DIR / "src")}
schema = subprocess.check_output([sys.executable, "-c", "from arxiv_pipeline.contracts import SCHEMA_VERSION; print(SCHEMA_VERSION)"], cwd=REPO_DIR, env=PIPELINE_ENV, text=True).strip()
if schema != "1.2":
    raise RuntimeError(f"Cần schema 1.2, checkout hiện tại trả {schema}.")
print(json.dumps({"commit": COMMIT_SHA, "python": platform.python_version(), "os": platform.platform(), "cpu_count": os.cpu_count(), "psutil": importlib.metadata.version("psutil")}, indent=2))


In [ ]:
import hashlib
import shutil
import signal
import subprocess
import tempfile
import sys
from pathlib import Path


def run_cli(args, *, allowed=(0,)):
    """Run argv without a shell, stream progress and preserve nonzero outcomes."""
    with subprocess.Popen(args, cwd=REPO_DIR, env=PIPELINE_ENV, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1,
                          start_new_session=True) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            exit_code = process.wait()
        except KeyboardInterrupt:
            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=45)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if exit_code not in allowed:
        raise RuntimeError(f"Lệnh trả mã {exit_code}; đọc output trước khi tiếp tục.")
    return exit_code



def run_snapshot_cli(args):
    """Use the same configured file cap when exporting or reading snapshots."""
    if type(SNAPSHOT_MAX_FILES) is not int or SNAPSHOT_MAX_FILES < 1:
        raise ValueError("SNAPSHOT_MAX_FILES phải là số nguyên dương.")
    return run_cli([sys.executable, "-m", "arxiv_pipeline.snapshots", *args,
                    "--max-files", str(SNAPSHOT_MAX_FILES)])


def sha256_file(path):
    result = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            result.update(chunk)
    return result.hexdigest()


def copy_verified(source, destination, *, expected_sha256=None):
    """Copy to a temporary file, verify bytes, then publish; never overwrite a different archive."""
    source, destination = Path(source), Path(destination)
    expected = sha256_file(source) if expected_sha256 is None else expected_sha256
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        if sha256_file(destination) == expected:
            return destination
        raise FileExistsError(f"Không ghi đè file khác nội dung: {destination}")
    with tempfile.NamedTemporaryFile(dir=destination.parent, prefix=destination.name + ".", suffix=".part", delete=False) as stream:
        temporary = Path(stream.name)
    try:
        shutil.copyfile(source, temporary)
        if sha256_file(temporary) != expected:
            raise ValueError("Bản sao không khớp SHA-256; chưa công bố file đích.")
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    return destination


## 3. Xác nhận thư mục Drive chung

Chạy cell này để tự cấp quyền mount Drive trong Colab. Chỉ ghi snapshot vào thư mục đã chia sẻ; không đặt corpus/SQLite ở đây. Chạy `full` cần điền và kiểm tra thư mục chung trước.

Có thể chạy pilot với đầu ra local khi chưa điền Drive, nhưng chưa được coi là đã sao lưu. Việc mount/ghi Drive thực tế phải được kiểm tra trong phiên Colab của người chạy.


In [ ]:
TEAM_PATH = None
if TEAM_DRIVE_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
    TEAM_PATH = Path(TEAM_DRIVE_DIR)
    if not TEAM_PATH.resolve().is_relative_to(Path("/content/drive").resolve()) or not TEAM_PATH.is_dir():
        raise ValueError("Điền đường dẫn của thư mục Drive chung đã có sau mount.")
    with tempfile.NamedTemporaryFile(dir=TEAM_PATH, prefix=f"access-test-{OWNER}-", delete=True) as probe:
        probe.write(b"ok")
        probe.flush()
    print("Đã kiểm tra quyền ghi:", TEAM_PATH)
elif RUN_MODE == "full":
    raise ValueError("Điền TEAM_DRIVE_DIR và kiểm tra sao lưu ở pilot trước khi chạy full.")
else:
    print("Chỉ lưu local. Chưa có bản sao lưu Drive cho tới khi hoàn tất cell bàn giao.")


## 4. Crawl và đo cùng lúc

`pilot` chọn tối đa MAX_PAPERS trong MAX_CANDIDATES; `full` bỏ cả hai giới hạn. `resume` dùng lại cấu hình trong SQLite. Không ghi đè thư mục đo cũ.

Ngắt bằng nút Stop/Interrupt sẽ chuyển SIGINT tới benchmark wrapper để nó dừng runner và lưu số đo. Sau đó kiểm tra corpus và xuất snapshot; không reset VM trước khi sao lưu. Nếu cell bị ngắt, chạy các cell kiểm tra/bàn giao thủ công sau khi tiến trình đã dừng.


### Khôi phục từ gói bàn giao (tùy chọn)

Trong phiên mới, điền RESTORE_BUNDLE trỏ tới zip trên Drive, giữ RUN_ID/OWNER cũ, chọn resume và MEASUREMENT_ID mới. Cell dưới kiểm tra checksum, chỉ lấy snapshot từ zip và gọi restore; không sửa dữ liệu gốc trên Drive.


In [ ]:
if RESTORE_BUNDLE:
    import zipfile
    if RUN_MODE != "resume" or TEAM_PATH is None:
        raise ValueError("Khôi phục cần RUN_MODE=resume và Drive đã được mount.")
    bundle_source = Path(RESTORE_BUNDLE)
    checksum_source = bundle_source.with_suffix(".zip.sha256")
    expected = checksum_source.read_text().split()[0]
    if not re.fullmatch(r"[0-9a-f]{64}", expected):
        raise ValueError("Checksum bàn giao không đúng định dạng.")
    restore_dir = WORK_ROOT / "restored" / RUN_ID / MEASUREMENT_ID
    if restore_dir.exists():
        raise FileExistsError("Đích restore đã có; không ghi đè corpus cũ.")
    incoming = WORK_ROOT / "restore-input" / RUN_ID / MEASUREMENT_ID
    incoming.mkdir(parents=True, exist_ok=True)
    local_bundle = copy_verified(bundle_source, incoming / bundle_source.name, expected_sha256=expected)
    with zipfile.ZipFile(local_bundle) as archive:
        identity = json.loads(archive.read("run.json"))
        if identity["owner"] != OWNER or identity["run_id"] != RUN_ID or identity["is_mock"] is not False:
            raise ValueError("Gói bàn giao không khớp owner/run ID hoặc là mock.")
        snapshot = incoming / "snapshot.tar.gz"
        with archive.open("snapshot.tar.gz") as source, snapshot.open("wb") as destination:
            shutil.copyfileobj(source, destination, length=1024 * 1024)
    run_snapshot_cli(["verify", "--archive", str(snapshot)])
    run_snapshot_cli(["restore", "--archive", str(snapshot), "--destination", str(restore_dir)])
    CORPUS_PATH = str(restore_dir / "corpus")
    print("Corpus đã restore:", CORPUS_PATH)


In [ ]:
import shlex

ensure_release()
OUTPUT_ROOT = Path(CORPUS_PATH) if CORPUS_PATH else WORK_ROOT / RUN_ID
MEASUREMENT_ROOT = WORK_ROOT / "measurements" / RUN_ID / MEASUREMENT_ID
BENCHMARK_DIR = MEASUREMENT_ROOT / "benchmark"
STATISTICS_DIR = MEASUREMENT_ROOT / "statistics"
HANDOFF_DIR = WORK_ROOT / "handoffs" / RUN_ID / MEASUREMENT_ID
if MEASUREMENT_ROOT.exists() and any(MEASUREMENT_ROOT.iterdir()):
    raise FileExistsError("Đổi MEASUREMENT_ID để giữ nguyên số đo cũ.")
if RUN_MODE == "resume":
    saved = json.loads((OUTPUT_ROOT / "runs" / RUN_ID / "scope.json").read_text())
    if saved["run_id"] != RUN_ID or saved["is_mock"] is not False or {r["owner"] for r in saved["scope"]["ranges"]} != {OWNER}:
        raise ValueError("Corpus không khớp owner/run ID hoặc đang là mock.")
else:
    if OUTPUT_ROOT.exists() and any(OUTPUT_ROOT.iterdir()):
        raise FileExistsError("Output root đã có dữ liệu: chọn run ID mới hoặc dùng resume.")

runner = [sys.executable, "-m", "arxiv_pipeline.runner", "--run-id", RUN_ID, "--output-root", str(OUTPUT_ROOT)]
if RUN_MODE == "resume":
    runner += ["--resume"]
else:
    runner += ["--owner", OWNER]
    runner += ["--full-range"] if RUN_MODE == "full" else ["--max-candidates", str(MAX_CANDIDATES), "--max-papers", str(MAX_PAPERS)]
if STOP_AFTER_TASKS is not None:
    if type(STOP_AFTER_TASKS) is not int or STOP_AFTER_TASKS < 1:
        raise ValueError("STOP_AFTER_TASKS phải là số nguyên dương hoặc None.")
    runner += ["--stop-after-tasks", str(STOP_AFTER_TASKS)]
CACHE_MODE = "mixed" if RUN_MODE == "resume" else "cold"
command = [sys.executable, "-m", "arxiv_pipeline.benchmark", "--run-id", RUN_ID,
           "--output-root", str(OUTPUT_ROOT), "--output-dir", str(BENCHMARK_DIR),
           "--cache-mode", CACHE_MODE, "--cmd", shlex.join(runner)]
print(shlex.join(command))
crawl_exit_code = run_cli(command, allowed=(0, 2))


## 5. Kiểm tra kết quả và xuất thống kê

`validation.valid=true` xác nhận cấu trúc và artifacts nhất quán. `quality.valid=false` cần xem theo từng bài: PDF-only giữ bản ghi rỗng; lỗi mạng còn thiếu dữ liệu phải được thử lại và ghi nhận. Exit 0 không tự chứng minh đã crawl toàn dải.

Wall time là thời gian của lượt đo. Thời gian từng bài là khoảng từ task đầu đến cuối của bài trong lượt đó, có cả thời gian chờ giữa các công đoạn; tổng các khoảng có thể vượt wall time. Resume không chạy lại discovery sẽ ghi null, không phải 0 hay số của lượt trước.


In [ ]:
run_cli([sys.executable, "-m", "arxiv_pipeline.collection", "inspect", "--output-root", str(OUTPUT_ROOT), "--run-id", RUN_ID], allowed=(0, 2))
run_cli([sys.executable, "-m", "arxiv_pipeline.statistics", "--output-root", str(OUTPUT_ROOT), "--run-id", RUN_ID, "--output-dir", str(STATISTICS_DIR)])
stats = json.loads((STATISTICS_DIR / f"{RUN_ID}.json").read_text())
bench = json.loads((BENCHMARK_DIR / f"{RUN_ID}.json").read_text())
print(json.dumps({"scope": stats["scope"], "summary": stats["summary"], "benchmark": bench}, ensure_ascii=False, indent=2))


## 6. Đóng gói và sao lưu lên Drive chung

Cell này chỉ chạy sau khi runner đã dừng. Snapshot CLI kiểm tra corpus và checksum; file zip bàn giao chứa snapshot, statistics, benchmark/timing và lệnh chạy. Không đưa corpus hoặc SQLite lên Git.

Đầu ra Drive: `<thư mục chung>/<owner>/<run_id>/<measurement_id>/`. Nếu chưa cấu hình Drive, gói vẫn ở local và **chưa được sao lưu ngoài VM**. Không ghi đè bản khác nội dung; người nhận phải kiểm tra checksum và snapshot trước khi ghép.

Giới hạn file lấy từ `SNAPSHOT_MAX_FILES` cho cả export và verify; giá trị được ghi vào `run.json`. Leader dùng cùng `--max-files` khi đọc gói lớn.


In [ ]:
from datetime import datetime, timezone
import zipfile

head = ensure_release()
run_cli([sys.executable, "-m", "arxiv_pipeline.collection", "inspect", "--output-root", str(OUTPUT_ROOT), "--run-id", RUN_ID], allowed=(0, 2))
if HANDOFF_DIR.exists():
    raise FileExistsError("Đã có gói bàn giao cho measurement này; giữ nguyên và chọn measurement ID mới khi tạo gói mới.")
HANDOFF_DIR.mkdir(parents=True)
SNAPSHOT = HANDOFF_DIR / "snapshot.tar.gz"
run_snapshot_cli(["export", "--output-root", str(OUTPUT_ROOT),
         "--run-id", RUN_ID, "--destination", str(SNAPSHOT), "--snapshot-id", f"{RUN_ID}-{MEASUREMENT_ID}", "--exporter-commit", head])
run_snapshot_cli(["verify", "--archive", str(SNAPSHOT)])
for folder in [BENCHMARK_DIR, STATISTICS_DIR]:
    if not folder.is_dir():
        raise FileNotFoundError("Cần xuất benchmark và statistics trước khi tạo gói bàn giao.")
    shutil.copytree(folder, HANDOFF_DIR / folder.name)
metadata = {"owner": OWNER, "run_id": RUN_ID, "measurement_id": MEASUREMENT_ID,
            "pipeline_commit": head, "schema_version": stats["schema_version"], "is_mock": bench["is_mock"], "scope": stats["scope"],
            "command": command, "cache_mode": CACHE_MODE, "snapshot_limits": {"max_files": SNAPSHOT_MAX_FILES}, "created_at": datetime.now(timezone.utc).isoformat(),
            "environment": bench["environment"], "psutil": importlib.metadata.version("psutil")}
(HANDOFF_DIR / "run.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2))
BUNDLE = HANDOFF_DIR.parent / f"{RUN_ID}-{MEASUREMENT_ID}.zip"
with zipfile.ZipFile(BUNDLE, "x", compression=zipfile.ZIP_STORED) as archive:
    for path in sorted(HANDOFF_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(HANDOFF_DIR))
CHECKSUM = BUNDLE.with_suffix(".zip.sha256")
CHECKSUM.write_text(f"{sha256_file(BUNDLE)}  {BUNDLE.name}\n")
print("Gói local:", BUNDLE)
if TEAM_PATH is not None:
    destination = TEAM_PATH / OWNER / RUN_ID / MEASUREMENT_ID
    copy_verified(BUNDLE, destination / BUNDLE.name)
    copy_verified(CHECKSUM, destination / CHECKSUM.name)
    print("Đã chép và đối chiếu SHA-256 tại:", destination)
else:
    print("CHƯA SAO LƯU DRIVE. Điền TEAM_DRIVE_DIR, mount Drive rồi chép gói trước khi mất phiên VM.")


## 7. Khôi phục trong phiên mới

Sau khi mount Drive và lấy đúng commit, chép **cặp zip + checksum** về local, kiểm tra SHA-256 rồi giải nén vào thư mục bàn giao mới. Dùng `snapshots verify` và `restore` với đường dẫn đích chưa tồn tại; không crawl trực tiếp trong snapshot đã unpack để ghép dữ liệu.

Ví dụ CLI (thay đường dẫn và run ID thực tế):

```bash
PYTHONPATH=src python -m arxiv_pipeline.snapshots verify --archive /content/handoff/snapshot.tar.gz --max-files 500000
PYTHONPATH=src python -m arxiv_pipeline.snapshots restore --archive /content/handoff/snapshot.tar.gz --destination /content/arxiv-restored --max-files 500000
```

Corpus mới nằm ở `/content/arxiv-restored/corpus`. Để tiếp tục qua các cell trên, đặt `RUN_MODE="resume"`, giữ đúng `RUN_ID`, đổi `MEASUREMENT_ID` và điền `CORPUS_PATH` ở cell cấu hình bằng đường dẫn corpus đã restore. Không chỉ đổi tên hay di chuyển thư mục chứa SQLite bằng tay.

Đọc [README repo](https://github.com/ChickenDev-cyber/arxiv-data-science#chay-chinh-thuc) để biết cách leader ghép snapshot theo owner và chuyển JSON sang Jupyter Book. Chưa có bằng chứng chạy toàn dải hoặc thử Colab thật chỉ từ việc notebook build/test offline thành công.

Sau khi restore thành công, trước đợt resume kế tiếp trong cùng phiên: **dán đường dẫn corpus vừa in vào dòng CORPUS_PATH ở cell cấu hình**, đặt RESTORE_BUNDLE = "", rồi đổi MEASUREMENT_ID. Chỉnh trong cell, không chỉ gán trong bộ nhớ; chạy lại cấu hình có thể đặt CORPUS_PATH về rỗng hoặc nạp lại ZIP cũ. Giữ đường dẫn corpus restore cũ khi đổi measurement để tiếp tục dữ liệu mới nhất.
